# Fine-tune VideoMAE on ASL Dataset

This notebook fine-tunes a VideoMAE transformer on the [American Sign Language Dataset](https://huggingface.co/datasets/ZahidYasinMittha/American-Sign-Language-Dataset) from Hugging Face.

**Dataset Info:**
- 108,618 videos representing 2,207 ASL words
- Each word has minimum 30 videos

**This notebook filters to the MOST IMPORTANT words:**
- Option 1: Top N most common English words
- Option 2: Specific target words (e.g., your local dataset words)

**Requirements:**
- GPU runtime recommended (Runtime → Change runtime type → GPU)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.40.0 datasets accelerate evaluate huggingface_hub av scikit-learn pandas

## 2. Imports

In [ ]:
import os
import random
from typing import Dict, List

import evaluate
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download, list_repo_files
from PIL import Image
from torch.utils.data import Dataset
from torchvision.io import read_video
from transformers import (
    AutoModelForVideoClassification,
    Trainer,
    TrainingArguments,
    VideoMAEImageProcessor,
)

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Configuration

**Key settings:**
- `FILTER_MODE`: Choose how to select important words
- `TOP_N_WORDS`: If using frequency filter, how many top words
- `TARGET_WORDS`: If using specific words, list them here

In [ ]:
# ============== CONFIGURATION ==============

# Dataset
REPO_ID = "ZahidYasinMittha/American-Sign-Language-Dataset"
CSV_FILENAME = "Aslense Dataset.csv"

# ========== WORD FILTERING (IMPORTANT!) ==========
# Choose ONE of these filter modes:
#   "top_frequent"  - Use top N most common English words
#   "target_words"  - Use specific list of words
#   "all"           - Use all 2,207 words (not recommended)

FILTER_MODE = "target_words"  # <-- CHANGE THIS

# For "top_frequent" mode: how many top words to use
TOP_N_WORDS = 100

# For "target_words" mode: specific words to train on
# These are the 40 most common words (matching your local dataset)
TARGET_WORDS = [
    # Basic common words
    "the", "of", "and", "to", "a", "in", "for", "is", "on", "that",
    "by", "this", "with", "i", "you", "it", "not", "or", "be", "are",
    "from", "at", "as", "your", "all", "have", "new", "more", "an", "was",
    "we", "will", "home", "can", "us", "about", "if", "page", "my", "has",
    "search", "free", "but", "our", "one", "other", "do", "no", "information",
    "time", "they", "up", "what", "which", "their", "out",
    # Add more words as needed...
]

# ========== MODEL ==========
MODEL_CKPT = "MCG-NJU/videomae-base-finetuned-kinetics"

# ========== TRAINING ==========
NUM_FRAMES = 16              # Frames to sample per video
NUM_TRAIN_EPOCHS = 10        # Number of epochs
LEARNING_RATE = 5e-5         # Learning rate
WEIGHT_DECAY = 0.05          # Weight decay
BATCH_SIZE = 4               # Per-device batch size (reduce if OOM)
GRAD_ACCUM_STEPS = 4         # Gradient accumulation
TRAIN_SPLIT = 0.9            # Train/eval split

# Output
OUTPUT_DIR = "videomae-asl"

# Reproducibility
SEED = 42

# ============================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Config:")
print(f"  Model: {MODEL_CKPT}")
print(f"  Filter mode: {FILTER_MODE}")
print(f"  Epochs: {NUM_TRAIN_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 4. Load Dataset Labels

In [ ]:
print("Downloading CSV labels...")
csv_path = hf_hub_download(repo_id=REPO_ID, filename=CSV_FILENAME, repo_type="dataset")
df_full = pd.read_csv(csv_path)

print(f"\nFull dataset statistics:")
print(f"  Total videos: {len(df_full):,}")
print(f"  Unique words: {df_full['word'].nunique():,}")

## 5. Filter to Important Words ⭐

This is where we reduce the dataset to only the most important/frequent words.

In [ ]:
# Most common English words (based on frequency analysis)
# These words appear in most ASL vocabularies and are useful for real-world applications
COMMON_ENGLISH_WORDS = [
    "the", "of", "and", "to", "a", "in", "for", "is", "on", "that",
    "by", "this", "with", "i", "you", "it", "not", "or", "be", "are",
    "from", "at", "as", "your", "all", "have", "new", "more", "an", "was",
    "we", "will", "home", "can", "us", "about", "if", "page", "my", "has",
    "search", "free", "but", "our", "one", "other", "do", "no", "information",
    "time", "they", "up", "what", "which", "their", "out", "there", "only",
    "so", "his", "when", "who", "also", "now", "help", "get", "pm", "view",
    "first", "am", "been", "would", "how", "were", "me", "some", "these",
    "its", "like", "than", "into", "just", "over", "such", "year", "may",
    "after", "should", "any", "right", "see", "only", "his", "day", "most",
    # Common verbs
    "go", "make", "know", "take", "come", "want", "give", "use", "find", "tell",
    "ask", "work", "seem", "feel", "try", "leave", "call", "need", "become", "put",
    # Common nouns
    "people", "man", "woman", "child", "world", "life", "hand", "part", "place", "case",
    "week", "company", "system", "program", "question", "work", "government", "number",
    "night", "point", "home", "water", "room", "mother", "father", "family", "student",
    # Common adjectives
    "good", "great", "old", "big", "small", "long", "little", "own", "other", "right",
    "high", "different", "next", "early", "young", "important", "public", "bad", "same",
    # Question words
    "what", "where", "when", "why", "how", "who", "which",
    # Pronouns
    "he", "she", "they", "we", "you", "i", "it", "my", "your", "his", "her", "their", "our",
    # Common expressions
    "yes", "no", "please", "thank", "sorry", "hello", "bye", "ok",
]

# Get all words available in the HF dataset (lowercase for matching)
hf_words = set(df_full['word'].str.lower().unique())
print(f"Words available in HF dataset: {len(hf_words)}")

# Apply filter based on mode
if FILTER_MODE == "top_frequent":
    # Filter to top N common words that exist in HF dataset
    target_set = set(w.lower() for w in COMMON_ENGLISH_WORDS[:TOP_N_WORDS])
    selected_words = target_set & hf_words
    print(f"\nUsing top {TOP_N_WORDS} frequent words")
    
elif FILTER_MODE == "target_words":
    # Filter to specific target words
    target_set = set(w.lower() for w in TARGET_WORDS)
    selected_words = target_set & hf_words
    print(f"\nUsing {len(TARGET_WORDS)} target words")
    
else:  # "all"
    selected_words = hf_words
    print(f"\nUsing ALL words (not recommended - very slow!)")

# Filter dataframe
df = df_full[df_full['word'].str.lower().isin(selected_words)].copy()

print(f"\n{'='*50}")
print(f"FILTERED DATASET:")
print(f"  Selected words: {len(selected_words)}")
print(f"  Total videos: {len(df):,}")
print(f"  Videos per word: ~{len(df) // len(selected_words) if selected_words else 0}")
print(f"{'='*50}")

# Show which words were selected
print(f"\nWords selected ({len(selected_words)}):")
print(sorted(selected_words))

In [ ]:
print("Building filename to path mapping (this may take a moment)...")
files = list_repo_files(REPO_ID, repo_type="dataset")
filename_to_path = {}
for f in files:
    if f.endswith(".mp4"):
        basename = f.split("/")[-1]
        filename_to_path[basename] = f

print(f"Found {len(filename_to_path):,} video files in repo")

## 6. Build Label Mappings

In [ ]:
# Build label mappings from FILTERED words only
all_words = sorted(df["word"].str.lower().unique())
label2id = {w: i for i, w in enumerate(all_words)}
id2label = {i: w for w, i in label2id.items()}
num_labels = len(all_words)

# Normalize the word column to lowercase for consistency
df['word'] = df['word'].str.lower()

print(f"Number of classes: {num_labels}")
print(f"\nWords: {all_words}")

## 7. Prepare Train/Eval Split

In [ ]:
# Shuffle and split
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split_idx = int(len(df) * TRAIN_SPLIT)
train_df = df.iloc[:split_idx]
eval_df = df.iloc[split_idx:]

print(f"Train samples: {len(train_df):,}")
print(f"Eval samples: {len(eval_df):,}")
print(f"\nClass distribution in train set:")
print(train_df['word'].value_counts().head(10))

## 8. Define Dataset Class

In [ ]:
class ASLVideoDataset(Dataset):
    """Dataset that loads videos from HF Hub based on CSV labels."""

    def __init__(
        self,
        df: pd.DataFrame,
        filename_to_path: Dict[str, str],
        label2id: Dict[str, int],
        processor: VideoMAEImageProcessor,
        num_frames: int = 16,
    ):
        self.df = df.reset_index(drop=True)
        self.filename_to_path = filename_to_path
        self.label2id = label2id
        self.processor = processor
        self.num_frames = num_frames

        # Filter to only rows where video exists in repo
        valid_mask = self.df["videos"].isin(filename_to_path.keys())
        self.df = self.df[valid_mask].reset_index(drop=True)
        print(f"Dataset has {len(self.df)} valid samples")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        video_filename = row["videos"]
        word = row["word"]
        label = self.label2id[word]

        # Get repo path and download video
        repo_path = self.filename_to_path[video_filename]
        try:
            local_path = hf_hub_download(
                repo_id=REPO_ID, filename=repo_path, repo_type="dataset"
            )
            # Read video frames using torchvision
            video, _, _ = read_video(local_path, pts_unit="sec")
            total_frames = video.shape[0]

            if total_frames == 0:
                raise ValueError("Empty video")

            # Sample frames uniformly
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
            frames = video[indices].numpy()

            # Convert to PIL images for processor
            images = [Image.fromarray(f) for f in frames]

            # Process frames
            inputs = self.processor(images, return_tensors="pt")
            pixel_values = inputs["pixel_values"].squeeze(0)

        except Exception as e:
            print(f"Error loading video {video_filename}: {e}")
            pixel_values = torch.zeros((self.num_frames, 3, 224, 224))

        return {"pixel_values": pixel_values, "labels": torch.tensor(label)}


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """Custom collate function."""
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

## 9. Load Model & Processor

In [ ]:
print(f"Loading model: {MODEL_CKPT}")
processor = VideoMAEImageProcessor.from_pretrained(MODEL_CKPT)

model = AutoModelForVideoClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

print(f"Model loaded with {num_labels} output classes")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 10. Create Datasets

In [ ]:
print("Creating train dataset...")
train_dataset = ASLVideoDataset(
    train_df, filename_to_path, label2id, processor, NUM_FRAMES
)

print("\nCreating eval dataset...")
eval_dataset = ASLVideoDataset(
    eval_df, filename_to_path, label2id, processor, NUM_FRAMES
)

## 11. Setup Training

In [ ]:
# Metrics
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),  # Enable FP16 on GPU
    seed=SEED,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

print("Trainer configured!")
print(f"  FP16: {training_args.fp16}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Steps per epoch: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")

## 12. Train! 🚀

In [ ]:
print("=" * 60)
print(f"Starting fine-tuning on {num_labels} words...")
print(f"Training {len(train_dataset)} videos for {NUM_TRAIN_EPOCHS} epochs")
print("=" * 60)

trainer.train()

## 13. Evaluate

In [ ]:
print("Evaluating best checkpoint...")
results = trainer.evaluate()

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
for k, v in results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 14. Save Model

In [ ]:
import json

final_dir = os.path.join(OUTPUT_DIR, "final")
os.makedirs(final_dir, exist_ok=True)

# Save model and processor
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)

# Save label mappings
with open(os.path.join(final_dir, "label2id.json"), "w") as f:
    json.dump(label2id, f, indent=2)
with open(os.path.join(final_dir, "id2label.json"), "w") as f:
    json.dump(id2label, f, indent=2)

# Save config info
config_info = {
    "filter_mode": FILTER_MODE,
    "num_classes": num_labels,
    "words": all_words,
    "train_samples": len(train_dataset),
    "eval_samples": len(eval_dataset),
}
with open(os.path.join(final_dir, "training_config.json"), "w") as f:
    json.dump(config_info, f, indent=2)

print(f"\n✅ Model saved to: {final_dir}")
print(f"\nFiles saved:")
for f in os.listdir(final_dir):
    print(f"  - {f}")

## 15. (Optional) Download Model from Colab

In [ ]:
# Uncomment to zip and download the model (for Google Colab)
# !zip -r videomae-asl-final.zip {final_dir}
# from google.colab import files
# files.download('videomae-asl-final.zip')

## 16. (Optional) Test Inference

In [ ]:
# Test inference on a random sample
model.eval()
model.to(device)

# Get a sample
sample_idx = random.randint(0, len(eval_dataset) - 1)
sample = eval_dataset[sample_idx]
pixel_values = sample["pixel_values"].unsqueeze(0).to(device)
true_label = sample["labels"].item()

with torch.no_grad():
    outputs = model(pixel_values=pixel_values)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred_label = probs.argmax(-1).item()
    confidence = probs[0, pred_label].item()

print(f"True label: {id2label[true_label]}")
print(f"Predicted:  {id2label[pred_label]} ({confidence:.1%} confidence)")
print(f"Correct: {'✅' if true_label == pred_label else '❌'}")